# 1. Check GPU

In [ ]:
#@title 1. Check GPU
import subprocess, sys
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit(
        "NO GPU ATTACHED.\n"
        "Runtime > Change runtime type > Hardware accelerator > T4 GPU, then rerun.\n"
        "Without this the model loads onto CPU and each request takes minutes."
    )
print(out.stdout)


# 2. Mount Drive and cache weights there

In [ ]:
#@title 2. Mount Drive and cache weights there
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# A fresh Colab session otherwise re-downloads ~7GB of weights, which is
# several minutes of dead air before the demo can start.
HF_HOME = "/content/drive/MyDrive/hf-cache"
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

cached = list(Path(HF_HOME).glob("hub/models--Qwen*"))
print(f"HF_HOME = {HF_HOME}")
print("CACHE HIT - weights already on Drive" if cached else "CACHE MISS - first run will download ~7GB")


# 3. Clone the repo and install

This cell also removes Colab's preinstalled `torchao` 0.10.0 if present: nothing here uses it, but peft's LoRA loader probes for it and raises (instead of skipping) when it's older than peft requires, which breaks loading any LoRA-adapter model. Runs automatically every session - no manual step needed.

In [ ]:
#@title 3. Clone the repo and install
REPO_URL = "https://github.com/ziad7amoda/target-ocr-mvp.git"  #@param {type:"string"}
BRANCH = "master"  #@param {type:"string"}

import os, shutil
if os.path.exists("/content/app-repo"):
    shutil.rmtree("/content/app-repo")
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/app-repo
%cd /content/app-repo
!pip install -q -r requirements.txt -r requirements-gpu.txt
print("installed")

# Print exactly which commit is running. Every confusing session so far has
# ended in the same question - "is this even the code with the fix in it?" -
# and answering it should not require guesswork about which branch a Colab
# form field was left on, or whether this notebook is a stale copy saved to
# Drive months ago.
!git -C /content/app-repo log -1 --pretty="running %h on {BRANCH} - %s"

# --- self-heal: Colab's preinstalled torchao is too old for peft --------
# Colab preinstalls torchao 0.10.0. Nothing in this project uses torchao,
# but loading any LoRA-adapter model (e.g.
# NAMAA-Space/Qari-OCR-0.4.0-VL-4B-Instruct) makes transformers hand off to
# peft, whose LoRA loader walks a list of layer dispatchers. One of them,
# dispatch_torchao, calls is_torchao_available() - which *raises* instead
# of returning False when torchao is present but older than peft requires
# (>0.16.0), killing the load even though the dispatcher is irrelevant.
# Uninstalling torchao makes the check short-circuit on "not installed" and
# return False, so the irrelevant dispatcher is skipped. This must run
# every session because a fresh Colab VM always restores torchao 0.10.0.
#
# The removal is VERIFIED, not assumed. `pip uninstall` exits 0 when the
# package is absent from the environment that pip manages, so a zero exit
# says nothing about whether anything was actually removed - and Colab's
# sys.executable and its `pip` do not always own the same site-packages.
# An earlier version of this cell trusted the exit code, printed "torchao
# removed", and let cell 4 die three minutes later on the very ImportError
# this exists to prevent.
import subprocess
import sys

if not os.path.exists("/content/app-repo/scripts/torchao_selfheal.py"):
    raise RuntimeError(
        f"The clone of branch '{BRANCH}' has no scripts/torchao_selfheal.py, so "
        "this is code from before the Colab dependency fixes landed. Either "
        "BRANCH above points at the wrong branch, or this notebook is an old "
        "copy saved in your Drive.\n\n"
        "Fix: set BRANCH to the branch carrying the fixes, or re-open the "
        "notebook from GitHub, then re-run this cell."
    )

from scripts.torchao_selfheal import decide_action, get_installed_version, uninstall_commands

_version = get_installed_version("torchao")
_action = decide_action(_version)

if _action == "skip-absent":
    print("torchao not installed - no action needed.")
elif _action == "skip-ok":
    print(f"torchao {_version} already satisfies peft's requirement - no action needed.")
else:
    print(
        f"torchao {_version} detected - nothing in this project uses it, but "
        "peft's LoRA loader probes for it and raises (instead of skipping) "
        "when it's older than 0.16.0, which would break loading any "
        "LoRA-adapter model. Removing it..."
    )
    for _command in uninstall_commands(sys.executable):
        subprocess.run(_command, capture_output=True, text=True)
        # Re-check rather than trust the exit code - see the note above.
        _version = get_installed_version("torchao")
        if _version is None:
            print("torchao removed - LoRA-adapter models will now load correctly.")
            break
    else:
        raise RuntimeError(
            f"torchao {_version} could not be removed automatically. Loading a "
            "LoRA-adapter model would fail on an ImportError from peft.\n\n"
            "Fix: run `!pip uninstall -y torchao` in a new cell, then "
            "Runtime > Restart session, then re-run cells 1-5.\n\n"
            "Stopping here rather than at the end of a multi-minute model load."
        )


# 4. Start the server and wait for the model

**Rule: if you are changing `MODEL_ID` from what it was the last time this cell ran, do Runtime -> Restart session first.** A model already loaded in this kernel cannot be freed while a server thread from a previous run of this cell is still holding it, so re-running the cell without restarting tries to fit two models in GPU memory at once and runs out of memory. This is the single most common mistake when comparing models back-to-back. The cell below now checks for this itself and will stop you with clear instructions if you forget - but restarting first avoids the wait.

`MODEL_ID` and `LOAD_IN_8BIT` below default to `MBZUAI/AIN` (Arabic-specialised, 7B, loaded in 8-bit to fit a 16GB T4). If AIN underperforms or is too slow during a live demo, the fast fallback is switching these two `#@param` fields — `MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"` and `LOAD_IN_8BIT = False` — and rerunning this cell, not editing code.

`PROMPT_STYLE` selects which field-extraction prompt is sent. `strict` is tuned to Qwen2.5-VL-3B and encodes fixes for several bugs observed only on that model; `natural` is minimal, has no model-specific coaching, and is the fair choice when comparing models against each other.

**Timing:** a cold cache (no prior Drive cache from cell 2) downloads roughly 15GB of AIN weights and then quantises them to 8-bit on load — realistically 15-25 minutes end to end. A warm Drive cache from an earlier session cuts this to a couple of minutes. `READY_TIMEOUT_MIN` below defaults to 30; raise it if you are on a slow connection. The cell prints a progress line (elapsed time, cache size on disk) while it waits, and fails immediately — not after the timeout — if the server thread actually crashes, rather than just loading slowly.

In [ ]:
#@title 4. Start the server and wait for the model
MODEL_ID = "MBZUAI/AIN"  #@param {type:"string"}
LOAD_IN_8BIT = True  #@param {type:"boolean"}
PROMPT_STYLE = "natural"  #@param ["natural", "strict", "transcribe"]
RECOVER_ARABIC_FROM_TRANSCRIPT = True  #@param {type:"boolean"}
# When extraction returns null for the Arabic name or place of birth,
# re-read just those from a transcription pass. Costs a second
# inference, but only on the cards that need it - a model that reads
# the Arabic properly never triggers it. Turn this off to measure a
# model's raw extraction ability with nothing propping it up.
MAX_VISION_TOKENS = 1280  #@param {type:"integer"}
# The processor downscales the card until it fits this many vision
# tokens (each covers a 28x28 patch), so it sets the resolution the
# model actually sees. 1280 is the Qwen2.5-VL default and is where
# every measurement so far was taken. It matters because the Arabic
# name is the smallest printed text on the card: raise this if a model
# reads the digits and dates correctly but returns null for the Arabic
# fields, which is what "too small to commit to" looks like from the
# outside. Costs prefill time and VRAM, not decode.
READY_TIMEOUT_MIN = 30  #@param {type:"integer"}
# A cold cache means downloading ~15GB of AIN weights, then quantising them
# to 8-bit on load - realistically 15-25 minutes. A warm Drive cache (cell
# 2) cuts this to a couple of minutes. Raise this if your connection is slow.

# --- pre-flight: refuse to load a second model on top of one already ----
# --- resident in this kernel ---------------------------------------------
# Changing MODEL_ID above and re-running this cell does NOT unload the
# previous model: the earlier server thread (started later in this same
# cell) is still alive and still holds a reference to it, so the new load
# just tries to allocate on top of the old one and the T4 runs out of
# memory. This has to run before app.main - and therefore the model/engine
# code - is imported, and before a new server thread is started.
import gc

_torch = None
try:
    import torch as _torch
except Exception as _torch_import_error:
    print(f"NOTE: could not import torch ({_torch_import_error}); skipping GPU pre-flight guard.")

if _torch is not None and _torch.cuda.is_available():
    _GUARD_THRESHOLD_BYTES = 500 * 1024 * 1024  # 500MB - "a model is resident"

    _prior_thread_alive = "server_thread" in globals() and globals()["server_thread"].is_alive()
    _before_bytes = _torch.cuda.memory_allocated()

    if _prior_thread_alive or _before_bytes > _GUARD_THRESHOLD_BYTES:
        # Try a clean release first: drop this cell's own objects from a
        # previous run, then ask the allocator to give memory back. This
        # cannot free memory held by a still-running server thread (its own
        # call stack keeps the model alive no matter what we do to these
        # globals), but it does recover memory left behind by a run that
        # never got a server going - e.g. only PROMPT_STYLE changed and
        # nothing here actually needs a restart.
        for _name in ("app", "server_thread", "_serve", "_startup_error", "_thread_error"):
            globals().pop(_name, None)
        gc.collect()
        _torch.cuda.empty_cache()
        _after_bytes = _torch.cuda.memory_allocated()

        if _prior_thread_alive or _after_bytes > _GUARD_THRESHOLD_BYTES:
            _gb_held = _after_bytes / (1024 ** 3)
            raise RuntimeError(
                f"{_gb_held:.2f} GiB is still allocated on the GPU by this "
                "process from a model loaded earlier in this session"
                + (
                    ", and its server thread is still running.\n"
                    if _prior_thread_alive
                    else ".\n"
                )
                + "Python cannot unload a model that a running server thread "
                "still holds a reference to, so re-running this cell just "
                "tries to load a second model on top of the first and runs "
                "out of GPU memory.\n\n"
                "Fix: Runtime -> Restart session, then re-run cells 1-5.\n"
                "Nothing re-downloads - the model cache lives on Drive (cell 2)."
            )

        _reclaimed_gb = (_before_bytes - _after_bytes) / (1024 ** 3)
        print(
            f"Reclaimed {_reclaimed_gb:.2f} GiB of GPU memory left over from a "
            "previous run of this cell - continuing without a restart."
        )

# --- print the resolved configuration before loading anything -----------
# Cheap, and it catches the other common mix-up: assuming a #@param edit
# took effect when this cell was not actually re-run.
print(f"MODEL_ID      = {MODEL_ID}")
print(f"LOAD_IN_8BIT  = {LOAD_IN_8BIT}")
print(f"PROMPT_STYLE  = {PROMPT_STYLE}")
print(f"VISION TOKENS = {MAX_VISION_TOKENS} (max)")
print(f"AR RECOVERY   = {RECOVER_ARABIC_FROM_TRANSCRIPT}")
if _torch is not None and _torch.cuda.is_available():
    _free_bytes, _total_bytes = _torch.cuda.mem_get_info()
    print(
        f"GPU memory    = {_free_bytes / 1024**3:.2f} GiB free / "
        f"{_total_bytes / 1024**3:.2f} GiB total"
    )
else:
    print("GPU memory    = unavailable (torch/CUDA not detected)")

import os

# Must be set BEFORE app.main (and therefore app.config) is imported, since
# Settings() reads the environment at import time.
os.environ["MODEL_ID"] = MODEL_ID
os.environ["LOAD_IN_8BIT"] = str(LOAD_IN_8BIT)
os.environ["PROMPT_STYLE"] = PROMPT_STYLE
os.environ["MAX_PIXELS"] = str(MAX_VISION_TOKENS * 28 * 28)
os.environ["RECOVER_ARABIC_FROM_TRANSCRIPT"] = str(RECOVER_ARABIC_FROM_TRANSCRIPT)

import logging
import threading
import time
import traceback
from pathlib import Path

import requests
import uvicorn

from app.main import app

# --- capture a crash instead of just timing out -------------------------
# The model loads inside FastAPI's lifespan startup, which runs inside
# uvicorn's event loop in this background thread. uvicorn swallows a
# failed lifespan startup itself (it logs the exception and exits the
# lifespan cleanly instead of letting it escape uvicorn.run()), so a bare
# try/except around uvicorn.run() would NOT see a bad model id, an OOM, or
# a bitsandbytes/dtype error - it would look identical to "still loading"
# and burn the whole timeout. Capture the traceback uvicorn already logs
# via a handler, and also catch what does propagate out of uvicorn.run()
# itself (e.g. sys.exit() on startup failure, or a port already in use).
_startup_error = {"traceback": None}


class _LifespanErrorCapture(logging.Handler):
    def emit(self, record):
        if record.exc_info:
            _startup_error["traceback"] = "".join(
                traceback.format_exception(*record.exc_info)
            )


logging.getLogger("uvicorn.error").addHandler(_LifespanErrorCapture())

_thread_error = {"traceback": None}


def _serve():
    try:
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    except BaseException:
        # BaseException, not Exception: a failed lifespan startup makes
        # uvicorn call sys.exit(), which raises SystemExit out of
        # uvicorn.run() itself.
        _thread_error["traceback"] = traceback.format_exc()


server_thread = threading.Thread(target=_serve, daemon=True)
server_thread.start()


def _cache_dir():
    # HF_HOME is set by cell 2 (Drive cache) - do not hardcode a Drive path
    # here so this also works if that cell is skipped or changed.
    home = os.environ.get("HF_HOME")
    return Path(home) if home else None


def _cache_size_bytes():
    """Best-effort size of the HF cache on disk, so the user can see the
    download growing instead of staring at a blank cell."""
    cache_dir = _cache_dir()
    if cache_dir is None or not cache_dir.exists():
        return None
    total = 0
    for p in cache_dir.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            continue
    return total


def _fmt_bytes(n):
    if n is None:
        return "unknown (cache dir not created yet)"
    size = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.1f}{unit}"
        size /= 1024
    return f"{size:.1f}TB"


def _crash_message():
    tb = _startup_error["traceback"] or _thread_error["traceback"]
    if tb:
        return tb
    return "(server thread exited but no traceback was captured)"


start = time.time()
start_size = _cache_size_bytes()
deadline = start + READY_TIMEOUT_MIN * 60
ready = False

while time.time() < deadline:
    if _startup_error["traceback"] or _thread_error["traceback"] or not server_thread.is_alive():
        print()
        raise RuntimeError(
            "Server crashed while loading the model - see traceback below:\n\n"
            + _crash_message()
        )

    try:
        h = requests.get("http://127.0.0.1:8000/api/health", timeout=5).json()
        if h.get("loaded"):
            print()
            print(h)
            ready = True
            break
    except Exception:
        pass

    elapsed_min = (time.time() - start) / 60
    print(
        f"\rwaiting for model... {elapsed_min:.1f} min elapsed, "
        f"cache on disk: {_fmt_bytes(_cache_size_bytes())}          ",
        end="",
        flush=True,
    )
    time.sleep(5)

if not ready:
    end_size = _cache_size_bytes()
    growing = (
        start_size is not None and end_size is not None and end_size > start_size
    )
    print()
    raise RuntimeError(
        f"Model did not become ready within {READY_TIMEOUT_MIN} minutes.\n"
        f"Cache dir was {'still growing' if growing else 'NOT growing'} "
        f"({_fmt_bytes(start_size)} -> {_fmt_bytes(end_size)}).\n\n"
        "What to do next:\n"
        "  1. Run `python scripts/bringup.py <image>` in a local/Colab "
        "terminal - it loads the model synchronously, outside a background "
        "thread, and will print the real error instead of hanging.\n"
        "  2. For a fast fallback, set MODEL_ID back to "
        "\"Qwen/Qwen2.5-VL-3B-Instruct\" and LOAD_IN_8BIT = False above, "
        "then rerun this cell."
    )


# 5. Open the public HTTPS tunnel

In [ ]:
#@title 5. Open the public HTTPS tunnel
# Quick tunnel rather than ngrok: no account, no auth token, one less thing
# to fail live.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, threading, time

url = None
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def _watch():
    global url
    for line in proc.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m and not url:
            url = m.group(0)

threading.Thread(target=_watch, daemon=True).start()
for _ in range(60):
    if url:
        break
    time.sleep(1)

print("\n" * 2 + "=" * 72)
print("   OPEN THIS URL:")
print(f"   {url}")
print("=" * 72 + "\n" * 2)
print("Camera capture needs HTTPS, which this tunnel provides.")


# 6. Smoke test before sharing your screen

This cell needs an image you supply. No sample photo ships in the repo (`eval/samples/*.jpg` is gitignored). Upload a card photo to this Colab session and set `IMAGE` below to its path, or run the sample generator in `eval/make_samples.py` after installing the Noto Naskh Arabic font.

In [ ]:
#@title 6. Smoke test before sharing your screen
# NOTE: this cell's output contains extracted field values (name, ID
# number, dates, ...). That is the point of a smoke test, but it also
# means those values persist inside a saved .ipynb - clear this cell's
# output before saving or committing the notebook.
import json, os, time, requests

IMAGE = "/content/app-repo/eval/samples/synthetic_01.jpg"  #@param {type:"string"}

if not os.path.exists(IMAGE):
    print("*" * 72)
    print("! NO SAMPLE IMAGE FOUND.")
    print(f"! {IMAGE} does not exist - no sample card ships in the repo")
    print("! (eval/samples/*.jpg is gitignored).")
    print("!")
    print("! To run this smoke test, either:")
    print("!   1. Upload a card photo to this Colab session (folder icon")
    print("!      on the left) and set IMAGE above to its path, or")
    print("!   2. Install the Noto Naskh Arabic font and run the sample")
    print("!      generator (eval/make_samples.py) to create one.")
    print("*" * 72)
else:
    t0 = time.time()
    r = requests.post(
        "http://127.0.0.1:8000/api/extract",
        files={"image": open(IMAGE, "rb")},
        # AIN in 8-bit is slower than the ~10s the old default budgeted
        # for - 7B, and int8 decode carries overhead - so give it real
        # headroom rather than a tight timeout.
        timeout=600,
    )
    print(f"HTTP {r.status_code} in {time.time() - t0:.1f}s")
    body = r.json()
    print(json.dumps(body["fields"], indent=2, ensure_ascii=False))
    print(f"agreement {body['agreement']}  elapsed_ms {body['elapsed_ms']}")